# 잡코리아 인재검색 & 입사제안
셀1 → 셀2 → 셀3 → (로그인 필요시 셀4) → 셀5 → 셀6 → 셀7 → 셀8 → 셀9

In [5]:
# ── [셀1] 라이브러리 & 설정 ────────────────────────────
import os, time
from pathlib import Path
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

_cwd = Path(os.getcwd())
for _p in [_cwd / '.env', _cwd / '10.rpa/70.webs/recruit/.env']:
    if _p.exists():
        load_dotenv(_p, override=True)
        print(f'.env: {_p}')
        break

JOBKOREA_ID = os.getenv('JOBKOREA_ID', '')
JOBKOREA_PW = os.getenv('JOBKOREA_PW', '')
PROFILE_DIR = str(Path.home() / '.jobkorea_profile')

assert JOBKOREA_ID, '❌ JOBKOREA_ID 없음 — .env 확인'
assert JOBKOREA_PW, '❌ JOBKOREA_PW 없음 — .env 확인'
print(f'✅ ID: {JOBKOREA_ID[:3]}*** | 프로필: {PROFILE_DIR}')

.env: d:\drive_files\10.worksfree\10.rpa\70.webs\recruit\.env
✅ ID: nam*** | 프로필: C:\Users\USER\.jobkorea_profile


In [6]:
# ── [셀2] Chrome 브라우저 시작 ─────────────────────────
import json

try:
    driver.quit()
    time.sleep(2)
    print('이전 브라우저 종료')
except:
    pass

# ── 프로필 잠금 파일 정리 ──
_lock = Path(PROFILE_DIR) / 'SingletonLock'
if _lock.exists():
    _lock.unlink()
    print('⚠️  SingletonLock 삭제')
    time.sleep(1)

# ── "페이지 복원" 팝업 차단 ──────────────────────────────
# Chrome Preferences 파일의 종료 상태를 Normal로 수정
# (이 파일에 Crashed 상태가 남아있으면 Chrome이 복원 팝업을 띄움)
_prefs_path = Path(PROFILE_DIR) / 'Default' / 'Preferences'
if _prefs_path.exists():
    try:
        _prefs = json.loads(_prefs_path.read_text(encoding='utf-8'))
        _prefs.setdefault('profile', {}).update({
            'exit_type': 'Normal',
            'exited_cleanly': True,
        })
        _prefs_path.write_text(json.dumps(_prefs), encoding='utf-8')
        print('✅ Preferences 복원 상태 초기화 (복원 팝업 차단)')
    except Exception as e:
        print(f'⚠️  Preferences 수정 실패: {e}')

options = Options()
options.page_load_strategy = 'eager'
options.add_argument(f'--user-data-dir={PROFILE_DIR}')
options.add_argument('--profile-directory=Default')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--start-maximized')
options.add_argument('--no-first-run')
options.add_argument('--no-default-browser-check')
options.add_experimental_option('prefs', {
    'profile.exit_type': 'Normal',
    'profile.exited_cleanly': True,
})

print('🚀 Chrome 시작 중...')
service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
except Exception as e:
    print(f'⚠️  1차 실패 → 3초 후 재시도')
    if _lock.exists(): _lock.unlink()
    time.sleep(3)
    driver = webdriver.Chrome(service=service, options=options)

driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument',
    {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'})
driver.maximize_window()
print('✅ 브라우저 시작 완료')

이전 브라우저 종료
✅ Preferences 복원 상태 초기화 (복원 팝업 차단)
🚀 Chrome 시작 중...
✅ 브라우저 시작 완료


In [7]:
# ── [셀3] 메인 접속 & 팝업 닫기 & 로그인 상태 확인 ────
import random

driver.execute_script('window.stop()')
time.sleep(random.uniform(0.5, 1.5))

try:
    driver.get('https://www.jobkorea.co.kr/')
    time.sleep(random.uniform(2.5, 4.0))
except TimeoutException:
    driver.execute_script('window.stop()')
    print('⚠️  페이지 로드 타임아웃 → 강제 중단 후 진행')
    time.sleep(random.uniform(0.5, 1.5))

try:
    driver.switch_to.alert.dismiss()
    print('  alert 닫음')
except:
    pass

_closed = 0
for sel in ['button[aria-label="닫기"]', '.popup-close', '.btn-close',
            '.modal-close', '#modalClose', 'button.close']:
    for btn in driver.find_elements(By.CSS_SELECTOR, sel):
        try:
            if btn.is_displayed():
                btn.click()
                time.sleep(random.uniform(0.3, 0.8))
                _closed += 1
        except:
            pass
if _closed:
    print(f'  팝업 {_closed}개 닫음')

page_src = driver.page_source
if '로그아웃' in page_src or '마이페이지' in page_src or 'MyPage' in page_src:
    logged_in = True
    print('✅ 로그인 유지 → [셀4] 건너뛰고 [셀5] 실행')
else:
    logged_in = False
    print('🔑 로그인 필요 → [셀4] 실행')

print(f'   페이지: {driver.title}')


🔑 로그인 필요 → [셀4] 실행
   페이지: 잡코리아 | 대한민국 대표 구인구직, 취업 정보 & 채용 플랫폼


In [ ]:
# ── [셀4] 로그인 or 인재검색 이동 ──────────────────────
if logged_in:
    # 이미 로그인 → 인재검색/인재정보 메뉴 클릭
    TALENT_XPATHS = [
        '/html/body/div[3]/header/div[1]/div/div/div[2]/div/nav/ul/li[8]/a',
        '//a[normalize-space(text())="인재검색"]',
        '//a[normalize-space(text())="인재정보"]',
        '//a[contains(text(),"인재검색")]',
        '//a[contains(text(),"인재정보")]',
        '//header//a[contains(text(),"인재")]',
        '//nav//a[contains(text(),"인재")]',
    ]
    clicked = False
    for xp in TALENT_XPATHS:
        try:
            talent_menu = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, xp)))
            label = talent_menu.text.strip()
            time.sleep(random.uniform(0.5, 1.5))
            talent_menu.click()
            time.sleep(random.uniform(1.5, 3.0))
            driver.execute_script('window.stop()')
            print(f'✅ "{label}" 클릭 → 이동 완료 | URL: {driver.current_url}')
            clicked = True
            break
        except:
            continue

    if not clicked:
        print('⚠️  인재검색/인재정보 메뉴 클릭 전부 실패 → URL 직접 이동')
        try:
            driver.get('https://www.jobkorea.co.kr/corp/person/find')
            time.sleep(random.uniform(1.5, 3.0))
            driver.execute_script('window.stop()')
            print(f'✅ 인재검색 URL 직접 이동 | URL: {driver.current_url}')
        except Exception as e:
            print(f'⚠️  URL 직접 이동 실패: {e}')

else:
    # ── 로그인 페이지로 직접 이동 ──────────────────────
    try:
        driver.get('https://www.jobkorea.co.kr/User/LogOn/LogOn')
        time.sleep(random.uniform(2.0, 3.5))
        driver.execute_script('window.stop()')
    except TimeoutException:
        driver.execute_script('window.stop()')
    print(f'   로그인 페이지 | URL: {driver.current_url}')

    # ── 기업회원 탭 클릭 (여러 XPath 시도) ────────────
    corp_tab_found = False
    for xp in [
        '//*[@id="devMemTab"]/li[2]/a',
        '//*[@id="devMemTab"]/li[2]',
        '//a[contains(text(),"기업회원")]',
        '//li[contains(text(),"기업회원")]',
        '//button[contains(text(),"기업회원")]',
        '//*[contains(@class,"tab") and contains(text(),"기업")]',
    ]:
        try:
            el = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, xp)))
            time.sleep(random.uniform(0.5, 1.0))
            el.click()
            time.sleep(random.uniform(0.8, 1.5))
            print(f'→ 기업회원 탭 클릭 성공: {xp}')
            corp_tab_found = True
            break
        except:
            continue

    if not corp_tab_found:
        # 기업회원 탭 없음 → 상단 메뉴 로그인 링크 직접 클릭
        FALLBACK_LOGIN_XPATH = '//*[@id="wrap"]/div[2]/div[1]/div[2]/div[1]/ul/li[5]/a'
        try:
            el = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, FALLBACK_LOGIN_XPATH)))
            time.sleep(random.uniform(0.5, 1.0))
            el.click()
            time.sleep(random.uniform(1.5, 2.5))
            print(f'→ 기업회원 탭 없음 → 상단 로그인 링크 클릭 | URL: {driver.current_url}')
        except Exception as e:
            print(f'⚠️  fallback 클릭도 실패: {e}')
            print(f'   현재 페이지 input 목록:')
            for inp in driver.find_elements(By.XPATH, '//input'):
                print(f'     id={inp.get_attribute("id")} name={inp.get_attribute("name")} type={inp.get_attribute("type")}')

    # ── ID / PW 입력 ────────────────────────────────────
    id_field = None
    for xp in ['//*[@id="M_ID"]', '//input[@name="M_ID"]',
                '//input[@type="text"]', '//input[@name="uId"]']:
        try:
            id_field = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, xp)))
            print(f'→ ID 입력창: {xp}')
            break
        except:
            continue

    if id_field:
        time.sleep(random.uniform(0.4, 1.0))
        id_field.clear()
        id_field.send_keys(JOBKOREA_ID)
    else:
        print('❌ ID 입력창 없음 — 브라우저에서 수동 입력 필요')

    pw_field = None
    for xp in ['//*[@id="M_PWD"]', '//input[@name="M_PWD"]',
                '//input[@type="password"]', '//input[@name="pwd"]']:
        try:
            pw_field = driver.find_element(By.XPATH, xp)
            print(f'→ PW 입력창: {xp}')
            break
        except:
            continue

    if pw_field:
        time.sleep(random.uniform(0.5, 1.5))
        pw_field.clear()
        pw_field.send_keys(JOBKOREA_PW)
        print('→ ID/PW 입력 완료')

    # ── 로그인 제출 ────────────────────────────────────
    time.sleep(random.uniform(0.5, 1.5))
    submitted = False
    for xp in [
        '//*[@id="login-form"]/fieldset/section[3]/button',
        '//button[@type="submit"]',
        '//input[@type="submit"]',
        '//button[contains(text(),"로그인")]',
    ]:
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, xp))).click()
            submitted = True
            print(f'→ 로그인 제출: {xp}')
            break
        except:
            continue
    if not submitted:
        print('❌ 로그인 버튼 없음')

    time.sleep(random.uniform(2.5, 4.0))
    try:
        driver.switch_to.alert.dismiss()
    except:
        pass

    current_url = driver.current_url.lower()
    logged_in   = 'logon' not in current_url and 'login' not in current_url
    print(f'{"✅ 로그인 성공" if logged_in else "❌ 로그인 실패"} | URL: {driver.current_url}')


In [ ]:
# ── [셀5] 인재검색 페이지 이동 ────────────────────────
# 현재 페이지 로딩 중단 (무한로딩 방지)
driver.execute_script('window.stop()')
time.sleep(0.5)

wait = WebDriverWait(driver, 10)

# 헤더 "인재검색" 메뉴 클릭
try:
    talent_menu = wait.until(EC.element_to_be_clickable((By.XPATH,
        '/html/body/div[3]/header/div[1]/div/div/div[2]/div/nav/ul/li[8]/a')))
    talent_menu.click()
    print('→ 인재검색 메뉴 클릭')
    time.sleep(2)
    # 클릭 후 새 페이지가 또 무한로딩하면 중단
    driver.execute_script('window.stop()')
    time.sleep(0.5)
except TimeoutException:
    print('⚠️  메뉴 클릭 타임아웃 → URL로 직접 이동')
    try:
        driver.get('https://www.jobkorea.co.kr/Recruit/Co_Read/C/personsearch')
    except TimeoutException:
        driver.execute_script('window.stop()')
        print('⚠️  URL 이동도 타임아웃 → 강제 중단 후 진행')
except Exception as e:
    print(f'⚠️  메뉴 클릭 실패: {e} → URL로 직접 이동')
    try:
        driver.get('https://www.jobkorea.co.kr/Recruit/Co_Read/C/personsearch')
    except TimeoutException:
        driver.execute_script('window.stop()')

# 팝업 닫기
for sel in ['button[aria-label="닫기"]', '.popup-close', '.btn-close', '.modal-close']:
    for btn in driver.find_elements(By.CSS_SELECTOR, sel):
        try:
            if btn.is_displayed():
                btn.click()
                time.sleep(0.3)
        except:
            pass

print(f'✅ 페이지: {driver.title}')
print(f'   URL   : {driver.current_url}')

In [ ]:
# 셀 6) 검색 조건 입력
from httpx import TimeoutException
import time

wait = WebDriverWait(driver, 10)

# ── 나이 조건 (30~61) ─────────────────────────────────
try:
    age_filter = wait.until(EC.element_to_be_clickable((By.XPATH,
        '//*[@id="dvSideFilter"]/ul/li[2]/strong')))
    age_filter.click()
    print('→ 나이 조건 클릭')
    age_start = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="txtAgeStart"]')))
    age_start.clear()
    age_start.send_keys('30')
    age_end = driver.find_element(By.XPATH, '//*[@id="txtAgeEnd"]')
    age_end.clear()
    age_end.send_keys('61')
    btn_age = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="btnAgeSearch"]')))
    btn_age.click()
    print('→ 나이 검색 버튼 클릭 (30~61)')
except TimeoutException:
    print('⚠️  나이 조건 타임아웃')

# ── 구직 상태 (전체 3개) ──────────────────────────────
try:
    wait.until(EC.element_to_be_clickable((By.XPATH,
        '//*[@id="dvSideFilter"]/ul/li[3]'))).click()
    print('→ 구직 상태 클릭')
    for idx in [1, 2, 3]:
        wait.until(EC.element_to_be_clickable((By.XPATH,
            f'//*[@id="dvJobStatus"]/label[{idx}]'))).click()
except TimeoutException:
    print('⚠️  구직 상태 타임아웃')

# ── 최근 활동일 (3개월 이내) ──────────────────────────
try:
    wait.until(EC.element_to_be_clickable((By.XPATH,
        '//*[@id="dvSideFilter"]/ul/li[6]'))).click()
    print('→ 최근 활동일 클릭')
    wait.until(EC.element_to_be_clickable((By.XPATH,
        '//*[@id="dvUpdateDt"]/label[5]'))).click()
except TimeoutException:
    print('⚠️  최근 활동일 타임아웃')

# ── 지역 (서울 전지역 + 경기 전지역) ─────────────────
REGION_BTN   = '//*[@id="dvKeyword"]/div[1]/div[2]/button[2]'
SEOUL        = '//*[@id="ulWorkingAreaIn"]/li[1]/label/span'
SEOUL_ALL    = '//*[@id="ulWorkingAreaLocalIn"]/li[1]/label/span'
GYEONGGI     = '//*[@id="ulWorkingAreaIn"]/li[2]/label/span'
GYEONGGI_ALL = '//*[@id="ulWorkingAreaLocalIn"]/li[1]/label/span'

try:
    # 지역 패널 열기
    wait.until(EC.element_to_be_clickable((By.XPATH, REGION_BTN))).click()
    print('→ 지역 패널 열기')
    time.sleep(0.5)

    # 서울 → 서울 전지역
    wait.until(EC.element_to_be_clickable((By.XPATH, SEOUL))).click()
    print('→ 서울 클릭')
    time.sleep(0.3)
    wait.until(EC.element_to_be_clickable((By.XPATH, SEOUL_ALL))).click()
    print('→ 서울 전지역 클릭')
    time.sleep(0.3)

    # 경기 → 경기 전지역
    wait.until(EC.element_to_be_clickable((By.XPATH, GYEONGGI))).click()
    print('→ 경기 클릭')
    time.sleep(0.3)
    wait.until(EC.element_to_be_clickable((By.XPATH, GYEONGGI_ALL))).click()
    print('→ 경기 전지역 클릭')
    time.sleep(0.3)

    # 지역 패널 닫기
    wait.until(EC.element_to_be_clickable((By.XPATH, REGION_BTN))).click()
    print('→ 지역 패널 닫기')

except TimeoutException:
    print('⚠️  지역 조건 타임아웃')
except Exception as e:
    print(f'⚠️  지역 조건 실패: {e}')


In [ ]:
# ── [셀7] 이력 로드 & 월간 한도 설정 ─────────────────
import json, datetime
from datetime import date, timedelta

MONTHLY_LIMIT = 250      # ← 이달 발송 한도 (100 또는 250)
HISTORY_FILE  = Path(os.getcwd()) / 'recruit_history.json'

# 이력 파일 로드
if HISTORY_FILE.exists():
    history = json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
    print(f'📂 이력 파일 로드: {len(history)}건')
else:
    history = []
    print('📂 이력 없음 (첫 실행)')

# 3개월 이내 발송 URL 집합 (로컬 기준 중복 방지)
cutoff = date.today() - timedelta(days=90)
recently_sent = {
    h['url'] for h in history
    if h.get('url') and h.get('sent_at')
    and date.fromisoformat(h['sent_at']) > cutoff
}

# 이번 달 잔여 한도 계산
this_month      = date.today().strftime('%Y-%m')
month_sent_cnt  = sum(1 for h in history if h.get('sent_at','').startswith(this_month))
remaining_quota = max(0, MONTHLY_LIMIT - month_sent_cnt)

print(f'3개월 내 발송(로컬): {len(recently_sent)}건')
print(f'이번 달 ({this_month}): {month_sent_cnt}건 / {MONTHLY_LIMIT}건')
print(f'남은 한도: {remaining_quota}건')
if remaining_quota == 0:
    print('⚠️  이번 달 한도 소진')

📂 이력 없음 (첫 실행)
3개월 내 발송(로컬): 0건
이번 달 (2026-06): 0건 / 250건
남은 한도: 250건


In [ ]:
# ── [셀8] 제안 메시지 로드 (proposal_message.md) ───────
MSG_FILE = Path(os.getcwd()) / 'proposal_message.md'

# 파일이 없으면 기본 템플릿 생성 (이후 VSCode에서 직접 편집)
# if not MSG_FILE.exists():
#     MSG_FILE.write_text("""\
# 안녕하세요, {name}님!

# 삼성생명(주) 채용의뢰를 받은 용산HR 이인성 팀장입니다.

# {name}님의 경력이 법인영업(GFC) 포지션을 제안드리위해 연락드립니다.
# 기업 CEO 대상 세무·재무·리스크 컨설팅 전문직이며
# "교육의 삼성" 답계 2개월간의 집중 교육을 통해 전문가를 양성합니다.
# 기업 보험에 대한 기회 증가로 삼성 생명에서는 단체 보험 사업의
# 대대적인 확장을 계획하고 있으며 이 에대한 일환으로 다양한 인재를 모시고자 합니다.
                        
# 사업 설명회에 참석하시면 상세한 내용을 확인해보실 수 있습니다.
# 관심 있으시면 편하신 시간에 회신 부탁드립니다.

# 이인성 팀장 | 삼성생명 용산HR
# """, encoding='utf-8')
#     print(f'📝 {MSG_FILE} 생성 — VSCode에서 직접 편집 가능')

PROPOSAL_MESSAGE = MSG_FILE.read_text(encoding='utf-8').strip()
print(f'📝 메시지 로드: {MSG_FILE.name} ({len(PROPOSAL_MESSAGE)}자)')
# print('─' * 40)
# print(PROPOSAL_MESSAGE.format(name='홍길동'))
# print('─' * 40)

📝 메시지 로드: proposal_message.md (330자)


In [ ]:
# ── [셀9-1] "포지션 제안" 버튼 첫 번째 클릭 ────────────
# .tdPosition : 각 후보자 행의 버튼 열
# "포지션 제안" 텍스트인 버튼만 선별 → 첫 번째 클릭

wait = WebDriverWait(driver, 10)

btn_cells = driver.find_elements(By.CSS_SELECTOR, '.tdPosition')
print(f'후보자 행 수: {len(btn_cells)}')

target_btn = None
for i, cell in enumerate(btn_cells):
    try:
        btns = cell.find_elements(By.TAG_NAME, 'button')
        for btn in btns:
            txt = btn.text.strip()
            if txt == '포지션 제안':
                print(f'  [{i+1}] 포지션 제안 버튼 발견')
                target_btn = btn
                break
            else:
                print(f'  [{i+1}] 버튼 텍스트: "{txt}" → 스킵')
    except:
        pass
    if target_btn:
        break

if target_btn:
    target_btn.click()
    print('✅ 포지션 제안 버튼 클릭 완료')
else:
    print('⚠️  포지션 제안 버튼 없음 (전부 보낸제안 보기 상태)')


In [ ]:
# ── [셀9-2] 새 탭 전환 → 제안 히스토리 확인 → 포지션 제안 버튼 클릭 ──
import time, random

PROPOSAL_HIST_XPATH = '/html/body/div[1]/div[3]/div/div[1]/div/div[2]'
PROPOSAL_BTN_XPATH  = '/html/body/div[1]/div[3]/div/div[1]/div/button[1]'

wait = WebDriverWait(driver, 10)

# ① 새 탭 감지 & 전환
all_wins  = driver.window_handles
main_win  = all_wins[0]
new_wins  = [w for w in all_wins if w != main_win]

if not new_wins:
    print('⚠️  새 탭 없음 — 셀9-1을 먼저 실행하세요')
else:
    driver.switch_to.window(new_wins[-1])
    driver.execute_script('window.stop()')
    time.sleep(random.uniform(1.0, 2.0))
    print(f'✅ 새 탭 전환 | URL: {driver.current_url}')

    # ② 제안 히스토리 확인
    hist_text = ''
    try:
        hist_el   = driver.find_element(By.XPATH, PROPOSAL_HIST_XPATH)
        hist_text = hist_el.text.strip()
    except:
        pass

    print(f'📋 히스토리 원문: {repr(hist_text)}')

    # "제안 내역이 없습니다" 포함 → 실제 내역 없음
    has_history = hist_text and '제안 내역이 없습니다' not in hist_text

    if has_history:
        print('📋 제안 히스토리 있음 → 스킵')
        for line in hist_text.splitlines():
            if line.strip():
                print(f'   | {line.strip()}')
    else:
        print('📋 제안 내역 없음 → 포지션 제안 버튼 클릭')
        try:
            btn = wait.until(EC.element_to_be_clickable((By.XPATH, PROPOSAL_BTN_XPATH)))
            print(f'   버튼 텍스트: "{btn.text.strip()}"')
            btn.click()
            print('✅ 포지션 제안 버튼 클릭 완료')
        except Exception as e:
            print(f'❌ 버튼 클릭 실패: {e}')


In [ ]:
# ── [셀9-3] 제안 건수 차감 확인 팝업 → "확인" 버튼 클릭 ──
import time, random

CONFIRM_BTN_XPATH = '//*[@id="dev-send-seletor"]/div/div/div/a'

wait = WebDriverWait(driver, 10)

try:
    confirm = wait.until(EC.element_to_be_clickable((By.XPATH, CONFIRM_BTN_XPATH)))
    print(f'   팝업 버튼 텍스트: "{confirm.text.strip()}"')
    confirm.click()
    time.sleep(random.uniform(0.5, 1.0))
    print('✅ 확인 버튼 클릭 완료')
except Exception as e:
    print(f'❌ 확인 버튼 없음: {e}')

# JS alert 처리 (팝업 대신 alert로 뜨는 경우)
try:
    alert = driver.switch_to.alert
    print(f'   alert 텍스트: "{alert.text}"')
    alert.accept()
    print('✅ alert 확인 완료')
except:
    pass


In [ ]:
# ── [셀9-4] 포지션 제안 팝업 → 포지션 선택 ─────────────
# 구조 분석: li > input(radio/checkbox) + label > span
# → 텍스트 클릭이 아니라 input 엘리먼트 클릭이 실제 선택
import time, random

POSITION_INPUT_XPATH     = '//*[@id="posgtitle"]'
DROPDOWN_CONTAINER_XPATH = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/table/tbody/tr[2]/td/div[2]/div/span/div/div[2]/div'
POSITION_KEYWORD         = 'WF20260603'

wait = WebDriverWait(driver, 10)

try:
    inp = wait.until(EC.element_to_be_clickable((By.XPATH, POSITION_INPUT_XPATH)))
    inp.clear()
    inp.send_keys(POSITION_KEYWORD)
    print(f'→ 검색어 입력: {POSITION_KEYWORD}')
    time.sleep(random.uniform(1.2, 1.8))

    # WF20260603 포함된 li 안의 input(radio/checkbox) 클릭
    radio = driver.find_element(By.XPATH,
        f'{DROPDOWN_CONTAINER_XPATH}//li[contains(., "WF20260603")]/input')
    print(f'   input type: {radio.get_attribute("type")}')
    print(f'   input checked: {radio.get_attribute("checked")}')

    driver.execute_script("arguments[0].click();", radio)
    time.sleep(0.5)

    val = inp.get_attribute('value')
    print(f'   입력 필드 값: "{val}"')
    print('✅ input 클릭 완료')

except Exception as e:
    print(f'❌ 실패: {e}')


In [ ]:
# ── [셀9-5] 제안 내용 입력 ───────────────────────────────
import time

CONTENT_XPATH = '//*[@id="posg_cntnt"]'

wait = WebDriverWait(driver, 10)

try:
    ta = wait.until(EC.presence_of_element_located((By.XPATH, CONTENT_XPATH)))

    # 기존 내용 전체 삭제 후 입력
    ta.clear()
    driver.execute_script("arguments[0].value = '';", ta)
    time.sleep(0.3)

    ta.send_keys(PROPOSAL_MESSAGE)
    time.sleep(0.3)

    val = ta.get_attribute('value')
    print(f'✅ 내용 입력 완료 ({len(val)}자)')
    print(f'   첫 줄: "{val.splitlines()[0] if val else ""}"')

except Exception as e:
    print(f'❌ 실패: {e}')


In [ ]:
# ── [셀9-6] 담당자 정보 입력 + 제안 보내기 ──────────────
import time, random

BASE = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/table/tbody'
MOBILE_DROPDOWN_BTN = f'{BASE}/tr[7]/td/div/div/span[1]/div/div[1]/button'
SUBMIT_XPATH  = '//*[@id="modalOfferPopup"]/div[2]/div/div[1]/div[2]/div/fieldset/div/button[2]'
CONFIRM_POPUP = '//*[@id="modalOfferPopup"]/div[2]/div/div[2]/div/div/button'

PHONE = ['010', '4935', '7573']
EMAIL = ('insung.lee', 'samsung.com')

wait = WebDriverWait(driver, 10)

def fill_by_xpath(xpath, value, label=''):
    try:
        el = driver.find_element(By.XPATH, xpath)
        try:
            inp = el.find_element(By.XPATH, './/input')
        except:
            inp = el
        driver.execute_script("arguments[0].value='';", inp)
        inp.send_keys(value)
        print(f'→ {label}: "{value}"')
    except Exception as e:
        print(f'⚠️  {label} 실패: {e}')

def fill_by_id(fid, value, label=''):
    try:
        el = driver.find_element(By.ID, fid)
        driver.execute_script("arguments[0].value='';", el)
        el.send_keys(value)
        print(f'→ {label}: "{value}"')
    except Exception as e:
        print(f'⚠️  {label} 실패: {e}')

# ① 이름 / 부서명
fill_by_id('lb_Ofc_Man_Name', '이인성', '이름')
fill_by_id('lb_dept_name',    '용산HR',  '부서명')

# ② 전화번호 010-4935-7573
fill_by_xpath(f'{BASE}/tr[6]/td/div/div/span[1]', PHONE[0], '전화1')
fill_by_xpath(f'{BASE}/tr[6]/td/div/div/span[2]', PHONE[1], '전화2')
fill_by_xpath(f'{BASE}/tr[6]/td/div/div/span[3]', PHONE[2], '전화3')

# ③ 핸드폰 드롭다운 두 번째 항목(010) 선택
try:
    btn = driver.find_element(By.XPATH, MOBILE_DROPDOWN_BTN)
    btn.click()
    time.sleep(0.7)
    items = driver.find_elements(By.XPATH,
        f'{MOBILE_DROPDOWN_BTN}/following::ul[1]/li | '
        f'{MOBILE_DROPDOWN_BTN}/following::div[contains(@class,"list")][1]//li')
    if len(items) >= 2:
        target = items[1]
        try:
            radio = target.find_element(By.XPATH, './/input')
            driver.execute_script("arguments[0].click();", radio)
        except:
            driver.execute_script("arguments[0].click();", target)
        print(f'→ 핸드폰1: "{target.text.strip()}" 선택')
    time.sleep(0.3)
except Exception as e:
    print(f'⚠️  핸드폰 드롭다운 실패: {e}')

fill_by_id('GuinOfcMan_Entity_Mobile_No2', PHONE[1], '핸드폰2')
fill_by_id('GuinOfcMan_Entity_Mobile_No3', PHONE[2], '핸드폰3')

# ④ 이메일 insung.lee@samsung.com
fill_by_id('GuinOfcMan_Entity_Email1', EMAIL[0], '이메일1')
fill_by_id('GuinOfcMan_Entity_Email2', EMAIL[1], '이메일2')

# ⑤ 제안 보내기 버튼 클릭
try:
    submit_btn = wait.until(EC.element_to_be_clickable((By.XPATH, SUBMIT_XPATH)))
    print(f'\n→ 제안 보내기 버튼: "{submit_btn.text.strip()}"')
    submit_btn.click()
    time.sleep(random.uniform(1.0, 2.0))

    # 완료 팝업 확인 버튼 클릭
    try:
        confirm_btn = wait.until(EC.element_to_be_clickable((By.XPATH, CONFIRM_POPUP)))
        print(f'   완료 팝업 버튼: "{confirm_btn.text.strip()}"')
        confirm_btn.click()
        time.sleep(0.5)
    except Exception as e:
        print(f'⚠️  완료 팝업 없음: {e}')

    print('✅ 제안 보내기 완료')
except Exception as e:
    print(f'❌ 제안 보내기 실패: {e}')


In [ ]:
# ── [셀9-7] 새 탭 닫고 메인 창으로 복귀 ────────────────
import time

main_win = driver.window_handles[0]
current  = driver.current_window_handle

if current != main_win:
    driver.close()
    driver.switch_to.window(main_win)
    print(f'✅ 새 탭 닫음 → 메인 창 복귀 | URL: {driver.current_url}')
else:
    print('ℹ️  이미 메인 창입니다')


In [ ]:
# 팝업창에서 확인 버튼 누르기를 아래 xpath로 시도해야 해.
# //*[@id="dev-send-seletor"]/div/div/div/a

In [17]:
# ── [셀9-진단] 검색결과 페이지 셀렉터 파악 ─────────────
# 셀9 실행 전에 이 셀을 먼저 실행해서 실제 후보자 항목 셀렉터를 확인하세요.

print(f'현재 URL: {driver.current_url}')
print(f'페이지 제목: {driver.title}')
print()

# 후보자가 들어있을 것 같은 셀렉터 후보를 모두 시도
candidates_to_check = [
    '.resumeBox', '.person-list-item', '.list-item', 'li.item',
    '.devPersonList li', '.resultInfo', '#listPerson li',
    '.listType02 li', '.co_person li', '.list_person li',
    'ul.list li', 'div.list li', '.srchPerson li',
    '.personList li', 'li.person', '.resumeList li',
    '.srchList li', 'tr.person', '.tbl_list tr',
]

found = []
for sel in candidates_to_check:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        found.append((sel, len(els)))
        print(f'✅ {sel:<35} → {len(els)}개 발견')

if not found:
    print('❌ 후보 셀렉터 없음 — 페이지 소스에서 직접 탐색합니다.')
    print()
    # 이름처럼 보이는 텍스트가 있는 요소 탐색
    src = driver.page_source
    # 후보자 이름 패턴 (OO님 등)
    import re
    # li 또는 div 태그 클래스명 추출
    classes = re.findall(r'class="([^"]+)"', src)
    class_counts = {}
    for c in classes:
        for cls in c.split():
            class_counts[cls] = class_counts.get(cls, 0) + 1
    # 10번 이상 반복되는 클래스 (목록 항목일 가능성)
    repeated = sorted([(v, k) for k, v in class_counts.items() if v >= 10], reverse=True)
    print('10회 이상 반복 클래스 (후보 셀렉터):')
    for cnt, cls in repeated[:20]:
        print(f'  .{cls:<40} ({cnt}회)')

현재 URL: https://www.jobkorea.co.kr/corp/person/find
페이지 제목: 인재검색 - 직무 경험이 풍부한 우수 인재 | 잡코리아

❌ 후보 셀렉터 없음 — 페이지 소스에서 직접 탐색합니다.

10회 이상 반복 클래스 (후보 셀렉터):
  .js-skillSearch                           (617회)
  .js-kwrdSearch                            (444회)
  .keywordBox                               (268회)
  .dvResumeLink                             (200회)
  .careerIcon                               (200회)
  .userInfoBox                              (137회)
  .userInfo                                 (137회)
  .devButtonScrap                           (137회)
  .career                                   (137회)
  .keywordSkill                             (136회)
  .keywordJob                               (130회)
  .hashTagBox                               (130회)
  .title                                    (117회)
  .inner                                    (106회)
  .btnClose                                 (106회)
  .detail                                   (101회)
  .tdSummary                      

In [ ]:
# ── [셀30] 브라우저 종료 (선택) ───────────────────────
# 세션 유지를 원하면 이 셀 실행 X
# 완전히 끝냈을 때만 실행

# driver.quit()
# print('✅ 브라우저 종료')

print('브라우저 유지 중 (다음 실행 시 로그인 불필요)')
print('종료하려면 driver.quit() 주석 해제 후 실행')